# Regresja wieloraka — analiza na zbiorze Energy Efficiency

**Cel ćwiczenia:** Zbudować i ocenić model regresji wielorakiej przewidujący zapotrzebowanie na energię (Heating Load) na podstawie parametrów budynku. Notebook realizuje kroki: wczytanie danych, EDA, przygotowanie cech, dopasowanie modeli OLS, diagnostykę reszt, analizę wielokolinearności (VIF), selekcję cech i regularizację (Ridge/Lasso), walidację oraz zapis wyników.

Źródło danych: UCI Energy Efficiency Dataset (ENB2012_data.xlsx).

In [ ]:
# Importy i ustawienia wykresów
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

%matplotlib inline
sns.set(style='whitegrid')

In [ ]:
# Wczytanie danych z repozytorium UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
df = pd.read_excel(url)
df.columns = [
    'RelativeCompactness', 'SurfaceArea', 'WallArea', 'RoofArea',
    'OverallHeight', 'Orientation', 'GlazingArea',
    'GlazingAreaDistribution', 'HeatingLoad', 'CoolingLoad'
]

print("Rozmiar danych:", df.shape)
print(df.head())
print('\nInfo:')
df.info()


## Szybkie EDA

Sprawdzimy statystyki opisowe, brakujące wartości, oraz macierz korelacji dla cech i zmiennych docelowych.

In [ ]:
# Statystyki i brakujące wartości
print(df.describe().T)
print('\nBrakujące wartości:')
print(df.isnull().sum())

# Macierz korelacji
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Macierz korelacji')
plt.show()

# Pairplot (wybrane cechy)
sns.pairplot(df[['RelativeCompactness','SurfaceArea','OverallHeight','GlazingArea','HeatingLoad','CoolingLoad']])
plt.show()


## Przygotowanie cech: enkodowanie i skalowanie

Zakodujemy zmienne kategoryczne `Orientation` i `GlazingAreaDistribution` za pomocą `get_dummies` (drop_first=True). Przygotujemy macierz predyktorów dla modelu.

In [ ]:
# Enkodowanie zmiennych kategorycznych
df_prep = pd.get_dummies(df, columns=['Orientation','GlazingAreaDistribution'], drop_first=True)
print('Nowe wymiary:', df_prep.shape)
print(df_prep.columns.tolist())


In [ ]:
# Przygotowanie X i y dla HeatingLoad oraz podział na zbiory
features = [c for c in df_prep.columns if c not in ['HeatingLoad','CoolingLoad']]
X = df_prep[features]
y = df_prep['HeatingLoad']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train/Test shapes:', X_train.shape, X_test.shape)


In [ ]:
# Dopasowanie modelu OLS dla HeatingLoad
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)
model_h = sm.OLS(y_train, X_train_c).fit()
print(model_h.summary())


In [ ]:
# Predykcje i metryki na zbiorze testowym
y_pred = model_h.predict(X_test_c)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R^2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

# Wykres reszt vs dopasowane
residuals = y_test - y_pred
plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Wartości przewidywane')
plt.ylabel('Reszty')
plt.title('Reszty vs przewidywane')
plt.show()

# Histogram reszt i Q-Q plot
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
sns.histplot(residuals, kde=True)
plt.title('Histogram reszt')
plt.subplot(1,2,2)
sm.qqplot(residuals, line='45', fit=True)
plt.title('Q-Q plot reszt')
plt.tight_layout()
plt.show()


In [ ]:
# Testy diagnostyczne (na resztach treningowych)
resid_train = model_h.resid

# Shapiro-Wilka (normalność)
sh_stat, sh_p = stats.shapiro(resid_train)
print(f"Shapiro-Wilk: stat={sh_stat:.4f}, p={sh_p:.4g}")

# Breusch-Pagan (heteroskedastyczność)
lm, lm_pvalue, fvalue, f_pvalue = het_breuschpagan(resid_train, model_h.model.exog)
print(f"Breusch-Pagan p-value={lm_pvalue:.4g}")

# Durbin-Watson (autokorelacja reszt)
dw = durbin_watson(resid_train)
print(f"Durbin-Watson: {dw:.4f}")


In [ ]:
# Obliczanie VIF
X_vif = X_train_c.drop(columns=['const'], errors='ignore')
vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))


## Selekcja cech i regularizacja (Ridge, Lasso)

Zastosujemy skalowanie i GridSearchCV dla Ridge oraz Lasso, aby zobaczyć wpływ regularizacji na współczynniki.

In [ ]:
# GridSearch dla Ridge i Lasso (pipeline ze skalowaniem)
pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
params = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
gs_ridge = GridSearchCV(pipe, params, cv=5, scoring='neg_mean_squared_error')
gs_ridge.fit(X_train, y_train)
print('Best Ridge alpha:', gs_ridge.best_params_)

pipe_l = Pipeline([('scaler', StandardScaler()), ('model', Lasso(max_iter=5000))])
params_l = {'model__alpha': [0.001, 0.01, 0.1, 1.0]}
gs_lasso = GridSearchCV(pipe_l, params_l, cv=5, scoring='neg_mean_squared_error')
gs_lasso.fit(X_train, y_train)
print('Best Lasso alpha:', gs_lasso.best_params_)

# Współczynniki (na oryginalnych cechach)
best_ridge = gs_ridge.best_estimator_
best_lasso = gs_lasso.best_estimator_
coefs = pd.DataFrame({
    'feature': X_train.columns,
    'ridge_coef': best_ridge.named_steps['model'].coef_,
    'lasso_coef': best_lasso.named_steps['model'].coef_
}).sort_values('ridge_coef', key=abs, ascending=False)
print(coefs.head(15))


In [ ]:
# Walidacja krzyżowa dla modelu liniowego
lr = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
cv_scores = cross_val_score(lr, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print('CV RMSE (5-fold):', np.round(cv_rmse,3))
print('Średnie RMSE:', np.round(cv_rmse.mean(),3))


In [ ]:
## Model dla CoolingLoad

# Przygotowanie danych dla CoolingLoad
X_c = df_prep[features]
y_c = df_prep['CoolingLoad']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
Xc_train_c = sm.add_constant(Xc_train)
Xc_test_c = sm.add_constant(Xc_test)

model_c = sm.OLS(yc_train, Xc_train_c).fit()
print(model_c.summary())

# Metryki na zbiorze testowym
yc_pred = model_c.predict(Xc_test_c)
rmse_c = mean_squared_error(yc_test, yc_pred, squared=False)
mae_c = mean_absolute_error(yc_test, yc_pred)
r2_c = r2_score(yc_test, yc_pred)
print(f"CoolingLoad — R^2: {r2_c:.4f}, RMSE: {rmse_c:.4f}, MAE: {mae_c:.4f}")


In [ ]:
# Zapis modelu i predykcji
# Zapisz najlepszy model Ridge
joblib.dump(best_ridge, 'lab04/best_ridge_heating.joblib')

# Zapis predykcji dla HeatingLoad
pred_df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})
pred_df.to_csv('lab04/heating_predictions.csv', index=False)
print('Zapisano model i predykcje w folderze lab04')


## Pytania kontrolne i wnioski

1. Co oznacza wysoka wartość współczynnika R²?
2. Jak interpretować ujemny współczynnik przy zmiennej `RelativeCompactness`?
3. Dlaczego warto analizować reszty modelu?
4. W jaki sposób można rozszerzyć analizę, aby objąć zmienną `CoolingLoad`?
5. Jakie ograniczenia ma model regresji wielorakiej w tym przypadku?

Wnioski:
- Model OLS dostarcza pierwszego wglądu w zależności między cechami a zapotrzebowaniem na energię.
- Regularizacja (Ridge/Lasso) ułatwia kontrolę nad współczynnikami i może zmniejszyć wariancję modelu.
- Analiza reszt i VIF jest niezbędna do weryfikacji założeń modelu i wykrywania wielokolinearności.

Następne kroki: log-transformacje zmiennych, inżynieria cech, modele nieliniowe (np. RandomForest) oraz kompleksowa walidacja czasowa, jeśli dane to wymagają.